In [1]:
from pathlib import Path
import pandas as pd

# Define dataset path
DATA_DIR = Path(
    r"C:\Users\bucco\Desktop\Laboratory of Data Science"
    r"\project\code_project\data\Project_Data"
)

FOLLOWS_PATH = DATA_DIR / "follows.json"

if not FOLLOWS_PATH.is_file():
    raise FileNotFoundError(
        f"File not found: {FOLLOWS_PATH}"
    )

# Load original dataset
df_follows_raw = pd.read_json(FOLLOWS_PATH)

# Create a working copy
df_follows_clean = df_follows_raw.copy()

print("Original dataset shape:", df_follows_raw.shape)

print("\nData types:")
display(df_follows_raw.dtypes.to_frame("dtype"))

print("\nDataset preview:")
display(df_follows_raw.head())

Original dataset shape: (4002115, 2)

Data types:


,dtype
follower,int64
followed,int64



Dataset preview:


,follower,followed
0,12475,22025
1,12475,22026
2,12475,22027
3,12475,22028
4,12475,17280


In [2]:
# Count duplicate relationships
exact_duplicates = df_follows_clean.duplicated(
    subset=["follower", "followed"]
).sum()

print("Duplicate relationships:", exact_duplicates)

# Remove duplicates
df_follows_clean = (
    df_follows_clean
    .drop_duplicates(
        subset=["follower", "followed"]
    )
    .reset_index(drop=True)
)

# Validate duplicate removal
remaining_duplicates = (
    df_follows_clean.duplicated(
        subset=["follower", "followed"]
    ).sum()
)

print("\nOriginal records:", len(df_follows_raw))
print("Duplicates removed:", exact_duplicates)
print("Cleaned records:", len(df_follows_clean))
print("Remaining duplicates:", remaining_duplicates)

assert remaining_duplicates == 0

Duplicate relationships: 99

Original records: 4002115
Duplicates removed: 99
Cleaned records: 4002016
Remaining duplicates: 0


In [3]:
# Convert identifiers to nullable integers
id_columns = ["follower", "followed"]

for col in id_columns:
    df_follows_clean[col] = pd.to_numeric(
        df_follows_clean[col],
        errors="coerce"
    ).astype("Int64")

# Check missing identifiers
missing_ids = df_follows_clean[id_columns].isna().sum()

# Check zero or negative identifiers
invalid_ids = (
    df_follows_clean[id_columns]
    .le(0)
    .sum()
)

# Build validation report
identifier_validation = pd.DataFrame({
    "Missing values": missing_ids,
    "Non-positive values": invalid_ids
})

print("IDENTIFIER VALIDATION")
print("-" * 40)

display(identifier_validation)

print("\nData types after conversion:")
display(df_follows_clean.dtypes.to_frame("dtype"))

# Identify relationships with invalid endpoints
invalid_relationships = (
    df_follows_clean[id_columns].isna().any(axis=1)
    | df_follows_clean[id_columns].le(0).any(axis=1)
)

print("\nRelationships with invalid identifiers:",
      invalid_relationships.sum())

# Inspect problematic records, if any
display(
    df_follows_clean.loc[
        invalid_relationships,
        id_columns
    ].head(10)
)

IDENTIFIER VALIDATION
----------------------------------------


,Missing values,Non-positive values
follower,0,0
followed,0,0



Data types after conversion:


,dtype
follower,Int64
followed,Int64



Relationships with invalid identifiers: 0


,follower,followed


In [4]:
# Identify self-follow relationships
self_follow_mask = (
    df_follows_clean["follower"]
    == df_follows_clean["followed"]
)

# Count self-follow relationships
self_follow_count = self_follow_mask.sum()

print("SELF-FOLLOW VALIDATION")
print("-" * 40)

print("Total relationships:", len(df_follows_clean))
print("Self-follow relationships:", self_follow_count)

print(
    "Self-follow percentage:",
    round(
        self_follow_count / len(df_follows_clean) * 100,
        4
    ),
    "%"
)

# Inspect self-follow relationships
print("\nSelf-follow records:")

display(
    df_follows_clean.loc[
        self_follow_mask,
        ["follower", "followed"]
    ].head(20)
)

SELF-FOLLOW VALIDATION
----------------------------------------
Total relationships: 4002016
Self-follow relationships: 0
Self-follow percentage: 0.0 %

Self-follow records:


,follower,followed


In [6]:
# Load cleaned Users dataset
USERS_CLEAN_PATH = DATA_DIR / "users_clean.csv"

if not USERS_CLEAN_PATH.is_file():
    raise FileNotFoundError(
        f"File not found: {USERS_CLEAN_PATH}"
    )

df_users_ref = pd.read_csv(
    USERS_CLEAN_PATH,
    usecols=["id_user"],
    dtype={"id_user": "Int64"}
)

# Extract valid user identifiers
valid_user_ids = pd.Index(
    df_users_ref["id_user"].dropna().unique()
)

print("Users loaded:", len(df_users_ref))
print("Unique valid user IDs:", len(valid_user_ids))

# Check foreign key references
invalid_follower = (
    ~df_follows_clean["follower"].isin(valid_user_ids)
)

invalid_followed = (
    ~df_follows_clean["followed"].isin(valid_user_ids)
)

unmatched_relationships = (
    invalid_follower | invalid_followed
)

# Identify distinct unknown users
unknown_user_ids = set(
    df_follows_clean.loc[
        invalid_follower, "follower"
    ]
).union(
    set(
        df_follows_clean.loc[
            invalid_followed, "followed"
        ]
    )
)

# Build referential integrity report
referential_summary = pd.DataFrame({
    "Check": [
        "Unknown follower references",
        "Unknown followed references",
        "Relationships with unknown users",
        "Distinct unknown user IDs"
    ],
    "Count": [
        invalid_follower.sum(),
        invalid_followed.sum(),
        unmatched_relationships.sum(),
        len(unknown_user_ids)
    ]
})

print("\nREFERENTIAL INTEGRITY — USERS")
print("-" * 45)

display(referential_summary)

print("\nUnknown user IDs:")
print(sorted(unknown_user_ids))

print("\nUnmatched relationships:")

display(
    df_follows_clean.loc[
        unmatched_relationships,
        ["follower", "followed"]
    ]
)

# Validate that the reference dataset was loaded
assert len(valid_user_ids) > 0, (
    "No valid user IDs were loaded."
)

FileNotFoundError: File not found: C:\Users\bucco\Desktop\Laboratory of Data Science\project\code_project\data\Project_Data\users_clean.csv

In [7]:
# Final Data Quality Assessment

# Check missing identifiers
missing_values = (
    df_follows_clean[["follower", "followed"]]
    .isna()
    .sum()
    .sum()
)

# Check non-positive identifiers
invalid_ids = (
    df_follows_clean[["follower", "followed"]]
    .le(0)
    .sum()
    .sum()
)

# Check duplicate relationships
duplicate_relationships = (
    df_follows_clean.duplicated(
        subset=["follower", "followed"]
    ).sum()
)

# Check self-follow relationships
self_follows = (
    df_follows_clean["follower"]
    == df_follows_clean["followed"]
).sum()

# Check referential integrity
unknown_followers = (
    ~df_follows_clean["follower"].isin(valid_user_ids)
).sum()

unknown_followed = (
    ~df_follows_clean["followed"].isin(valid_user_ids)
).sum()

unmatched_relationships = (
    ~df_follows_clean["follower"].isin(valid_user_ids)
    | ~df_follows_clean["followed"].isin(valid_user_ids)
).sum()

# Build final quality report
quality_report = pd.DataFrame({
    "Check": [
        "Missing identifiers",
        "Non-positive identifiers",
        "Duplicate relationships",
        "Self-follow relationships",
        "Unknown follower references",
        "Unknown followed references",
        "Relationships with unknown users"
    ],
    "Count": [
        missing_values,
        invalid_ids,
        duplicate_relationships,
        self_follows,
        unknown_followers,
        unknown_followed,
        unmatched_relationships
    ]
})

print("FINAL DATA QUALITY ASSESSMENT")
print("-" * 45)

display(quality_report)

print("\nDATASET COMPARISON")
print("-" * 45)

print("Original records:", len(df_follows_raw))
print("Duplicates removed:", exact_duplicates)
print("Final records:", len(df_follows_clean))

print("\nFinal dataset shape:", df_follows_clean.shape)

# Validate structural integrity
assert missing_values == 0
assert invalid_ids == 0
assert duplicate_relationships == 0
assert self_follows == 0

assert len(df_follows_clean) == (
    len(df_follows_raw) - exact_duplicates
)

assert df_follows_clean.columns.tolist() == [
    "follower", "followed"
]

print("\nAll internal data quality checks passed.")

FINAL DATA QUALITY ASSESSMENT
---------------------------------------------


,Check,Count
0,Missing identifiers,0
1,Non-positive identifiers,0
2,Duplicate relationships,0
3,Self-follow relationships,0
4,Unknown follower references,4002016
5,Unknown followed references,4002016
6,Relationships with unknown users,4002016



DATASET COMPARISON
---------------------------------------------
Original records: 4002115
Duplicates removed: 99
Final records: 4002016

Final dataset shape: (4002016, 2)

All internal data quality checks passed.


In [8]:
# Define output path
FOLLOWS_CLEAN_PATH = DATA_DIR / "follows_clean.csv"

# Preserve original schema
df_follows_final = df_follows_clean[
    ["follower", "followed"]
].copy()

# Export cleaned dataset
df_follows_final.to_csv(
    FOLLOWS_CLEAN_PATH,
    index=False
)

# Final checks
assert len(df_follows_final) == 4002016
assert df_follows_final.shape[1] == 2

print("EXPORT COMPLETED")
print("-" * 40)

print("File:", FOLLOWS_CLEAN_PATH)

print("\nFinal dataset shape:")
print(df_follows_final.shape)

print("\nFinal dataset preview:")
display(df_follows_final.head())

EXPORT COMPLETED
----------------------------------------
File: C:\Users\bucco\Desktop\Laboratory of Data Science\project\code_project\data\Project_Data\follows_clean.csv

Final dataset shape:
(4002016, 2)

Final dataset preview:


,follower,followed
0,12475,22025
1,12475,22026
2,12475,22027
3,12475,22028
4,12475,17280
